# StudyMate · Part 3 of 8: Feature Engineering and Search

**Assignment part:** Feature Engineering & Core Functionality  
**Group 7:** Sadia Saeed · Eva Abouabouharb · Neida Guzman

Turn text into features and find the right lesson: **TF-IDF** (word matching), **all-MiniLM-L6-v2 embeddings** (meaning matching), the **hybrid score** (30/70), the **cross-encoder reranker** score, and the **50/50 fused final score** (min-max normalized).

> 📖 **This notebook is for reading.** It is one section of the full notebook, with the real outputs kept. Each part uses results from the parts before it, so to **run** StudyMate, open [`FN_StudyMate_Group7_ITAI2377.ipynb`](FN_StudyMate_Group7_ITAI2377.ipynb) and use **Runtime → Run all**.

## 4. Search: Hybrid Retrieval + Reranker

**Purpose:** Find the best lessons for a student's question.

**How it works (two-stage retrieval):**
1. **Stage 1, hybrid search:** TF-IDF (word matching, 30%) + MiniLM embeddings (meaning matching, 70%) find the top 5 lessons.
2. **Stage 2, reranker:** A cross-encoder (`cross-encoder/ms-marco-MiniLM-L-6-v2`) reads the question together with each of the 5 lessons.
3. **Score fusion:** Both scores are scaled to 0–1 and mixed 50/50 to pick the final top 3.

**Settings** (30/70 hybrid, top-5 pool, 50/50 fusion) are based on our earlier testing.

**Output:** The top 3 lessons for 3 demo questions.

In [37]:
# CODE CELL 4 — SEARCH: HYBRID RETRIEVAL + RERANKER
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, CrossEncoder

texts = lib["search_text"].tolist()

# Stage 1: TF-IDF + embeddings
tfidf_vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
tfidf_matrix = tfidf_vectorizer.fit_transform(texts)
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
lesson_embeddings = embedding_model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)

# Stage 2: reranker
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def minmax(x):
    x = np.asarray(x, dtype=float)
    return (x - x.min()) / (x.max() - x.min()) if x.max() > x.min() else np.ones_like(x)

def hybrid_scores(question):
    tfidf = cosine_similarity(tfidf_vectorizer.transform([question]), tfidf_matrix).flatten()
    q_emb = embedding_model.encode([question], convert_to_numpy=True, normalize_embeddings=True)
    emb = cosine_similarity(q_emb, lesson_embeddings).flatten()
    return 0.30 * tfidf + 0.70 * emb

def search(question, top_n=3, pool=5, w_rerank=0.5):
    scores = hybrid_scores(question)
    top = np.argsort(-scores)[:pool]                                   # stage 1: top 5
    finalists = lib.iloc[top].copy()
    finalists["hybrid_score"] = scores[top]
    pairs = [(question, t) for t in finalists["search_text"]]
    finalists["rerank_score"] = reranker.predict(pairs)                # stage 2: careful judge
    finalists["final_score"] = ((1 - w_rerank) * minmax(finalists["hybrid_score"])
                                + w_rerank * minmax(finalists["rerank_score"]))
    return finalists.sort_values("final_score", ascending=False).head(top_n).reset_index(drop=True)

# Demo
for q in ["What is the difference between precision and recall?",
          "How does gradient descent help a model learn?",
          "Why does my model do well on training data but badly on new data?"]:
    print(f"\nQ: {q}")
    display(search(q)[["title", "topic", "final_score"]].round(3))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


Q: What is the difference between precision and recall?


,title,topic,final_score
0,F1 Score,Model Evaluation,1.000
1,Recall,Model Evaluation,0.869
2,Precision,Model Evaluation,0.833



Q: How does gradient descent help a model learn?


,title,topic,final_score
0,Gradient Descent,Training,1.000
1,Loss and Optimizers,Neural Networks,0.335
2,Backpropagation,Training,0.176



Q: Why does my model do well on training data but badly on new data?


,title,topic,final_score
0,Overfitting,Overfitting and Underfitting,1.000
1,Underfitting,Overfitting and Underfitting,0.411
2,Data Leakage,Overfitting and Underfitting,0.356
